# OSS Founder Departure Survival Analysis

## Experiment Overview

This notebook implements a survival analysis experiment that tests the **inverted-U hypothesis** between knowledge redundancy (KR) and open-source project survival after founder departure.

### Research Question
What determines whether an open-source project survives its founder stepping away?

### Hypothesis
Moderate knowledge redundancy (KR) optimizes project survival - too little means the founder is irreplaceable, too much means contributions are redundant.

### Methodology
- **Founder departure detection**: Using Avelino et al. (2019) 12-month threshold with gap detection
- **Survival measurement**: TFDD definition (3+ months without founder commits)
- **Knowledge Redundancy (KR)**: Computed using cosine similarity of file_count histograms across top contributors (fallback approach due to lack of file path data)
- **Statistical analysis**: Cox proportional hazards model, Kaplan-Meier curves, bootstrap confidence intervals

### Data
The experiment processes commit records from open-source repositories to detect founder departures and measure project survival.

## Install Dependencies

This cell installs the required packages. On Colab, core packages (numpy, pandas, etc.) are pre-installed and don't need to be installed. The `_pip()` helper function handles this automatically.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install)
_pip('loguru')
_pip('psutil')
_pip('lifelines')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

print("Dependencies installed successfully!")

## Imports and Setup

Import all required libraries and set up the environment. This is copied directly from the original `method.py` with minimal changes.

In [ ]:
from loguru import logger
from pathlib import Path
import json
import sys
import gc
import os
import resource
import numpy as np
import pandas as pd
from itertools import combinations
from datetime import datetime, timedelta
from collections import defaultdict
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Configure logging
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

# Set memory limits (conservative: 6GB for demo)
import psutil
_avail = psutil.virtual_memory().available
RAM_BUDGET = min(6 * 1024**3, _avail * 0.7)  # 6GB or 70% of available
resource.setrlimit(resource.RLIMIT_AS, (int(RAM_BUDGET * 1.5), int(RAM_BUDGET * 1.5)))

print("Imports and setup complete!")

## Data Loading

Load the demo data from GitHub (with local fallback). The `mini_demo_data.json` file contains a curated subset of 3 examples for quick demonstration.

In [ ]:
# GitHub URL for the demo data (will work after files are pushed to GitHub)
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-a68c06-knowledge-redundancy-predicts-oss/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

# Load the data
data = load_data()
print(f"Data loaded successfully!")
print(f"Number of datasets: {len(data.get('datasets', []))}")
if 'datasets' in data and len(data['datasets']) > 0:
    examples = data['datasets'][0].get('examples', [])
    print(f"Number of examples: {len(examples)}")

## Configuration

Define all tunable parameters. For this demo, we use **MINIMUM** values to ensure quick execution. These can be scaled up gradually for more meaningful results.

### Key Parameters:
- `departure_threshold_months`: Months of inactivity to consider founder departed (default: 12)
- `time_window_months`: Time window before departure to analyze contributor patterns (default: 24)
- `max_commits`: Maximum commits to process per repo (default: 5000, demo: 100)
- `n_bootstrap`: Number of bootstrap samples (default: 200, demo: 10)

In [ ]:
# CONFIGURATION - Set to MINIMUM values for demo

# Founder departure detection
DEPARTURE_THRESHOLD_MONTHS = 12  # Months of inactivity to consider departure

# Knowledge Redundancy computation
TIME_WINDOW_MONTHS = 24  # Time window before departure to analyze
MAX_COMMITS = 100  # Maximum commits to process (small for demo)

# Bootstrap analysis
N_BOOTSTRAP = 10  # Number of bootstrap samples (small for demo)

# Output settings
OUTPUT_DIR = "results"

print("Configuration set:")
print(f"  Departure threshold: {DEPARTURE_THRESHOLD_MONTHS} months")
print(f"  Time window: {TIME_WINDOW_MONTHS} months")
print(f"  Max commits: {MAX_COMMITS}")
print(f"  Bootstrap samples: {N_BOOTSTRAP}")

## Helper Functions

These functions are copied directly from `method.py` with minimal changes. They handle:
1. Parsing examples from the dataset
2. Grouping records by repository
3. Identifying founders
4. Detecting founder departures

In [ ]:
def parse_examples(examples):
    """Parse examples into structured format."""
    parsed = []

    for ex in examples:
        try:
            # Parse input JSON
            input_data = json.loads(ex['input'])

            # Extract fields
            record = {
                'repo_id': input_data.get('repo_id', ''),
                'repo_name': input_data.get('repo_name', ''),
                'author_login': input_data.get('author_login', ''),
                'is_founder': input_data.get('is_founder', False),
                'file_count': input_data.get('file_count', 0),
                'commit_sequence_num': input_data.get('commit_sequence_num', 0),
                'author_total_commits': input_data.get('author_total_commits', 0),
                'repo_total_commits': input_data.get('repo_total_commits', 0),
                'commit_timestamp': input_data.get('commit_timestamp', ''),
                'commit_sha': ex.get('metadata_commit_sha', ''),
                'output': ex.get('output', ''),
            }

            # Parse timestamp
            if record['commit_timestamp']:
                try:
                    # Handle ISO format with timezone
                    ts = record['commit_timestamp'].replace('Z', '+00:00')
                    record['datetime'] = datetime.fromisoformat(ts)
                except:
                    record['datetime'] = None

            parsed.append(record)

        except Exception as e:
            logger.debug(f"Failed to parse example: {e}")
            continue

    logger.info(f"Parsed {len(parsed)} valid records")
    return parsed


def group_by_repo(records):
    """Group records by repository."""
    repos = defaultdict(list)
    for record in records:
        repos[record['repo_id']].append(record)

    # Sort each repo's records by timestamp
    for repo_id in repos:
        repos[repo_id] = sorted(
            [r for r in repos[repo_id] if r.get('datetime')],
            key=lambda x: x['datetime']
        )

    logger.info(f"Grouped into {len(repos)} repositories")
    return repos


def identify_founder(repo_records):
    """Identify founder using multiple methods."""
    if not repo_records:
        return None

    # Method 1: Use is_founder flag if available
    founders = [r for r in repo_records if r.get('is_founder')]
    if founders:
        return founders[0]['author_login']

    # Method 2: Earliest commit author (first commit)
    if repo_records:
        return repo_records[0]['author_login']

    return None


def detect_founder_departure(repo_records, founder, departure_threshold_months=12):
    """Detect founder departure using Avelino et al. threshold.
    
    Also checks if there's a significant gap in founder's contributions,
    not just complete stop.
    """
    if not founder or not repo_records:
        return None, None

    # Get founder's commits
    founder_commits = [r for r in repo_records if r['author_login'] == founder]
    if not founder_commits:
        return None, None

    # Sort by timestamp
    founder_commits = sorted(founder_commits, key=lambda x: x['datetime'])

    # Last commit by founder
    last_commit = founder_commits[-1]
    last_commit_date = last_commit['datetime']

    # Check if 12+ months since last founder commit
    departure_threshold = last_commit_date + timedelta(days=departure_threshold_months * 30)

    # Get repo's last commit date
    repo_last_commit = max(r['datetime'] for r in repo_records)

    # Also check: is there a 6+ month gap in founder's contributions before the last commit?
    # This captures "reduced activity" departures
    if len(founder_commits) >= 2:
        gaps = []
        for i in range(1, len(founder_commits)):
            gap_days = (founder_commits[i]['datetime'] - founder_commits[i-1]['datetime']).days
            gaps.append(gap_days)
        
        max_gap = max(gaps)
        if max_gap >= 180:  # 6+ month gap
            # Find the date of the gap
            for i in range(1, len(founder_commits)):
                if (founder_commits[i]['datetime'] - founder_commits[i-1]['datetime']).days >= 180:
                    gap_date = founder_commits[i-1]['datetime']
                    # Use gap date as departure if it's earlier than last commit
                    if gap_date < last_commit_date:
                        logger.info(f"Founder gap departure detected: {gap_date}")
                        return founder, gap_date

    if repo_last_commit > departure_threshold:
        # Founder has been gone for 12+ months
        return founder, last_commit_date
    else:
        # Founder still active or recently active
        return None, None

print("Helper functions defined.")

## Core Analysis Functions

These functions implement the main analysis logic:
1. `compute_pseudo_kr`: Compute knowledge redundancy using file_count distributions
2. `measure_survival`: Measure project survival after founder departure
3. `compute_control_variables`: Compute control variables for regression

In [ ]:
def compute_pseudo_kr(repo_records, founder, departure_date, time_window_months=24, max_commits=5000):
    """Compute pseudo-Knowledge Redundancy using file_count patterns.

    Since we don't have file paths for Jaccard similarity, we use file_count
    distributions as a proxy. This measures the similarity in file modification
    patterns across contributors.

    Approach:
    1. Get top contributors (excluding founder post-departure)
    2. For each contributor, compute distribution of file_counts
    3. Compute pairwise similarity using cosine similarity of distributions
    4. Average to get project-level KR
    
    Args:
        max_commits: Maximum number of commits to use (sample if more)
    """
    if not departure_date or not repo_records:
        return None, None

    # Define time window before departure
    window_start = departure_date - timedelta(days=time_window_months * 30)

    # Get commits in time window
    window_commits = [
        r for r in repo_records
        if window_start <= r['datetime'] <= departure_date
    ]

    if not window_commits:
        return None, None
    
    # LIMIT COMMITS for performance (sample if too many)
    if len(window_commits) > max_commits:
        logger.info(f"Sampling {max_commits} from {len(window_commits)} commits for performance")
        import random
        random.seed(42)
        window_commits = random.sample(window_commits, max_commits)

    # Get top contributors by commit count (exclude founder post-departure)
    contributor_commits = defaultdict(list)
    for commit in window_commits:
        author = commit['author_login']
        if author == founder:
            # Only include founder commits before departure
            if commit['datetime'] <= departure_date:
                contributor_commits[author].append(commit)
        else:
            contributor_commits[author].append(commit)

    # Keep top 5 contributors
    top_contributors = sorted(
        contributor_commits.items(),
        key=lambda x: len(x[1]),
        reverse=True
    )[:5]

    if len(top_contributors) < 2:
        return None, None

    # Compute file_count distributions for each contributor
    contributor_distributions = {}
    for author, commits in top_contributors:
        file_counts = [c['file_count'] for c in commits if c['file_count'] > 0]
        if file_counts:
            # Create histogram (distribution) of file counts
            hist, _ = np.histogram(file_counts, bins=10, range=(0, max(file_counts)))
            contributor_distributions[author] = hist

    # Compute pairwise cosine similarity
    similarities = []
    for (auth1, dist1), (auth2, dist2) in combinations(contributor_distributions.items(), 2):
        # Cosine similarity
        dot_product = np.dot(dist1, dist2)
        norm1 = np.linalg.norm(dist1)
        norm2 = np.linalg.norm(dist2)

        if norm1 > 0 and norm2 > 0:
            sim = dot_product / (norm1 * norm2)
            similarities.append(sim)

    if not similarities:
        return None, None

    # Average pairwise similarity = Knowledge Redundancy
    kr = np.mean(similarities)
    kr_squared = kr ** 2

    return kr, kr_squared


def measure_survival(repo_records, departure_date, founder, observation_end_date=None):
    """Measure project survival after founder departure.

    Uses Avelino et al. (2019) TFDD definition:
    - Survives if new contributors join and project continues
    - More robust: check for commits 3+ months after departure

    Returns:
    - survived: binary (1 if survived, 0 if not)
    - survival_time: days from departure to first post-departure commit by NON-FOUNDER
    - censored: whether survival time is censored
    """
    if not departure_date or not repo_records or not founder:
        return None, None, None

    if observation_end_date is None:
        observation_end_date = max(r['datetime'] for r in repo_records)

    # Get commits 3+ months after departure by NON-FOUNDER contributors
    # This gives time for the project to actually "die" if it will
    three_months_after = departure_date + timedelta(days=90)
    
    post_departure = [
        r for r in repo_records
        if r['datetime'] > three_months_after and r['author_login'] != founder
    ]

    if not post_departure:
        # No post-departure commits by others after 3 months = did not survive
        # Censored at observation end
        survival_time = (observation_end_date - departure_date).days
        return 0, survival_time, 1

    # Sort by timestamp
    post_departure = sorted(post_departure, key=lambda x: x['datetime'])
    first_post_commit = post_departure[0]['datetime']

    # Project survived - compute time to first non-founder commit (from departure)
    all_post_departure = [
        r for r in repo_records
        if r['datetime'] > departure_date and r['author_login'] != founder
    ]
    all_post_departure = sorted(all_post_departure, key=lambda x: x['datetime'])
    
    if all_post_departure:
        survival_time = (all_post_departure[0]['datetime'] - departure_date).days
    else:
        survival_time = (observation_end_date - departure_date).days

    return 1, survival_time, 0


def compute_control_variables(repo_records, founder, departure_date):
    """Compute control variables for survival analysis."""
    if not repo_records:
        return {}

    # Project age at departure
    repo_created = min(r['datetime'] for r in repo_records)
    age_days = (departure_date - repo_created).days if departure_date else 0

    # Contributor count
    contributors = set(r['author_login'] for r in repo_records if r['datetime'] <= departure_date)
    contributor_count = len(contributors)

    # Total commits pre-departure
    pre_departure_commits = len([r for r in repo_records if r['datetime'] <= departure_date])

    # Bus factor approximation (simplified)
    # Using Avelino's insight: if top contributor has >50% commits, bus factor = 1
    commit_counts = defaultdict(int)
    for r in repo_records:
        if r['datetime'] <= departure_date:
            commit_counts[r['author_login']] += 1

    if commit_counts:
        max_contributions = max(commit_counts.values())
        bus_factor = 1 if max_contributions > pre_departure_commits * 0.5 else 2
    else:
        bus_factor = 1

    return {
        'project_age_days': age_days,
        'contributor_count': contributor_count,
        'total_commits_pre': pre_departure_commits,
        'bus_factor': bus_factor,
    }

print("Core analysis functions defined.")

## Statistical Analysis Functions

These functions perform the statistical analysis:
1. `run_survival_analysis`: Cox proportional hazards model
2. `run_kaplan_meier`: Kaplan-Meier survival curves
3. `bootstrap_confidence_intervals`: Bootstrap CI for effect sizes

In [ ]:
def run_survival_analysis(results_df):
    """Run Cox proportional hazards model to test inverted-U hypothesis.

    H0: KR^2 coefficient = 0 (no inverted-U)
    H1: KR^2 coefficient < 0 (inverted-U: moderate KR optimal)
    """
    try:
        from lifelines import CoxPHFitter
        from lifelines.utils import concordance_index

        logger.info("Running Cox proportional hazards model...")

        # Prepare data
        df = results_df.copy()

        # Remove rows with missing data
        df = df.dropna(subset=['survival_time', 'survived', 'kr', 'kr_squared'])

        if len(df) < 10:
            logger.warning("Insufficient data for Cox model")
            return None

        # Fit Cox model
        # Formula: survival_time ~ KR + KR^2 + controls
        cph = CoxPHFitter(penalizer=0.01)  # Small penalty for stability

        # Prepare covariates
        covariates = ['kr', 'kr_squared', 'bus_factor', 'contributor_count', 'project_age_days']
        X = df[covariates].copy()
        X = X.apply(pd.to_numeric, errors='coerce')

        T = df['survival_time'].values
        E = df['survived'].values

        cph.fit(X, duration_col=None, event_col=None, T=T, E=E)

        # Extract results
        results = {
            'cox_model_summary': cph.summary.to_dict() if hasattr(cph, 'summary') else {},
            'kr_coef': cph.params_['kr'] if 'kr' in cph.params_ else None,
            'kr_squared_coef': cph.params_['kr_squared'] if 'kr_squared' in cph.params_ else None,
            'kr_squared_p_value': cph.summary.loc['kr_squared', 'p'] if 'kr_squared' in cph.summary.index else None,
            'hazard_ratios': {k: np.exp(v) for k, v in cph.params_.items()},
            'concordance': cph.concordance_index_,
        }

        # Test inverted-U: KR^2 coefficient should be negative
        kr2_coef = results.get('kr_squared_coef')
        kr2_p = results.get('kr_squared_p_value')

        if kr2_coef is not None and kr2_p is not None:
            results['inverted_u_supported'] = kr2_coef < 0 and kr2_p < 0.05
            results['inverted_u_direction'] = 'negative' if kr2_coef < 0 else 'positive'

        logger.info(f"Cox model complete. Concordance: {results.get('concordance', 'N/A')}")
        logger.info(f"KR^2 coefficient: {kr2_coef:.4f}, p-value: {kr2_p:.4f}")

        return results

    except ImportError:
        logger.error("lifelines not installed. Cannot run Cox model.")
        return None
    except Exception as e:
        logger.error(f"Cox model failed: {e}")
        return None


def run_kaplan_meier(results_df):
    """Run Kaplan-Meier survival curves with log-rank test."""
    try:
        from lifelines import KaplanMeierFitter
        from lifelines.statistics import logrank_test

        logger.info("Running Kaplan-Meier analysis...")

        df = results_df.copy()
        df = df.dropna(subset=['survival_time', 'survived', 'kr'])

        if len(df) < 10:
            return None

        # Create KR tertiles
        df['kr_tertile'] = pd.qcut(df['kr'], q=3, labels=['low', 'medium', 'high'])

        km_results = {}

        # Fit KM for each tertile
        for tertile in ['low', 'medium', 'high']:
            subset = df[df['kr_tertile'] == tertile]
            if len(subset) < 3:
                continue

            kmf = KaplanMeierFitter()
            kmf.fit(subset['survival_time'], event_observed=subset['survived'])

            km_results[tertile] = {
                'survival_function': kmf.survival_function_.to_dict(),
                'median_survival_time': kmf.median_survival_time_,
                'n_observed': len(subset),
            }

        # Log-rank test (low vs high)
        if 'low' in km_results and 'high' in km_results:
            low_group = df[df['kr_tertile'] == 'low']
            high_group = df[df['kr_tertile'] == 'high']

            if len(low_group) >= 3 and len(high_group) >= 3:
                lr_test = logrank_test(
                    low_group['survival_time'], high_group['survival_time'],
                    event_observed_A=low_group['survived'], event_observed_B=high_group['survived']
                )
                km_results['logrank_test'] = {
                    'statistic': lr_test.test_statistic,
                    'p_value': lr_test.p_value,
                }

        logger.info("Kaplan-Meier analysis complete")
        return km_results

    except ImportError:
        logger.error("lifelines not installed. Cannot run Kaplan-Meier.")
        return None
    except Exception as e:
        logger.error(f"Kaplan-Meier failed: {e}")
        return None


def bootstrap_confidence_intervals(results_df, n_bootstrap=200):
    """Compute bootstrap confidence intervals for effect sizes."""
    logger.info(f"Running bootstrap with {n_bootstrap} resamples...")

    # Need at least 3 samples for tertiles
    if len(results_df) < 3:
        logger.warning("Insufficient data for bootstrap (< 3 samples)")
        return None

    bootstrap_samples = []

    for i in range(n_bootstrap):
        # Resample with replacement
        sample = results_df.sample(n=len(results_df), replace=True)

        # Compute KR effect (difference in survival between tertiles)
        try:
            sample['kr_tertile'] = pd.qcut(sample['kr'], q=3, labels=['low', 'medium', 'high'], duplicates='drop')
        except:
            # If qcut fails, use simple median split
            median_kr = sample['kr'].median()
            sample['kr_tertile'] = sample['kr'].apply(lambda x: 'low' if x < median_kr else 'high')
            sample['kr_tertile'] = sample['kr_tertile'].replace({'low': 'low', 'high': 'high'})

        # Get survival rates for low and high KR
        if 'low' in sample['kr_tertile'].values and 'high' in sample['kr_tertile'].values:
            low_survival = sample[sample['kr_tertile'] == 'low']['survived'].mean()
            high_survival = sample[sample['kr_tertile'] == 'high']['survived'].mean()

            if not np.isnan(low_survival) and not np.isnan(high_survival):
                bootstrap_samples.append({
                    'low_survival': low_survival,
                    'high_survival': high_survival,
                    'diff': high_survival - low_survival,
                })

    if not bootstrap_samples:
        return None

    # Compute 95% CI
    diffs = [s['diff'] for s in bootstrap_samples]
    ci_lower = np.percentile(diffs, 2.5)
    ci_upper = np.percentile(diffs, 97.5)

    results = {
        'bootstrap_n': len(bootstrap_samples),
        'survival_diff_mean': np.mean(diffs),
        'survival_diff_95ci': [ci_lower, ci_upper],
    }

    logger.info(f"Bootstrap complete. 95% CI for survival diff: [{ci_lower:.3f}, {ci_upper:.3f}]")
    return results

print("Statistical analysis functions defined.")

## Main Processing

This is the main processing loop that:
1. Parses the input data
2. Groups records by repository
3. Identifies founders and detects departures
4. Computes knowledge redundancy (KR)
5. Measures survival
6. Runs statistical analyses

In [ ]:
# Create output directory
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# Extract examples from the loaded data
if 'datasets' in data and len(data['datasets']) > 0:
    examples = data['datasets'][0].get('examples', [])
else:
    examples = []

print(f"Processing {len(examples)} examples...")

# Parse examples
parsed_records = parse_examples(examples)

# Group by repository
repos = group_by_repo(parsed_records)

print(f"Found {len(repos)} repositories")

# Process each repository
results = []

for repo_id, repo_records in repos.items():
    try:
        logger.info(f"Processing {repo_id} ({len(repo_records)} records)")
        
        # Identify founder
        founder = identify_founder(repo_records)
        if not founder:
            logger.warning(f"Could not identify founder for {repo_id}")
            continue
        
        logger.info(f"Founder: {founder}")
        
        # Detect founder departure
        founder_departed, departure_date = detect_founder_departure(
            repo_records, founder, DEPARTURE_THRESHOLD_MONTHS
        )
        
        if departure_date:
            logger.info(f"Founder {founder} departed on {departure_date}")
            
            # Compute pseudo-KR (knowledge redundancy)
            kr, kr_squared = compute_pseudo_kr(
                repo_records, founder, departure_date,
                TIME_WINDOW_MONTHS, MAX_COMMITS
            )
            if kr is None:
                logger.warning(f"Could not compute KR for {repo_id}")
                continue
            
            # Measure survival
            survived, survival_time, censored = measure_survival(
                repo_records, departure_date, founder
            )
            if survival_time is None:
                continue
            
            # Compute control variables
            controls = compute_control_variables(repo_records, founder, departure_date)
            
            # Store results for departure case
            result = {
                'repo_id': repo_id,
                'founder': founder,
                'departure_date': departure_date.isoformat(),
                'kr': kr,
                'kr_squared': kr_squared,
                'survived': survived,
                'survival_time': survival_time,
                'censored': censored,
                'has_departure': True,
                **controls,
            }
            results.append(result)
            
            logger.info(f"Repo {repo_id}: KR={kr:.3f}, Survived={survived}, Time={survival_time}d")
        else:
            # No departure detected - still include as example with output="no_departure"
            # Compute KR anyway for completeness
            kr, kr_squared = compute_pseudo_kr(
                repo_records, founder, repo_records[-1]['datetime'],
                TIME_WINDOW_MONTHS, MAX_COMMITS
            )
            
            if kr is not None:
                result = {
                    'repo_id': repo_id,
                    'founder': founder,
                    'departure_date': None,
                    'kr': kr,
                    'kr_squared': kr_squared,
                    'survived': None,  # No departure = no survival measurement
                    'survival_time': None,
                    'censored': None,
                    'has_departure': False,
                }
                results.append(result)
                logger.info(f"Repo {repo_id}: No departure, KR={kr:.3f}")
            
    except Exception as e:
        logger.error(f"Error processing {repo_id}: {e}")
        continue

logger.info(f"Processed {len(results)} repos with founder departure")
print(f"\nProcessing complete! Found {len(results)} results.")

## Results and Visualization

Display the results in a readable format and create visualizations to understand the relationship between knowledge redundancy and survival.

In [ ]:
# Convert results to DataFrame
if results:
    results_df = pd.DataFrame(results)
    
    # Display summary statistics
    print("="*60)
    print("EXPERIMENT RESULTS SUMMARY")
    print("="*60)
    
    # Filter to only repos with departure
    departure_results = results_df[results_df['has_departure'] == True]
    
    if len(departure_results) > 0:
        print(f"\nRepositories with founder departure: {len(departure_results)}")
        print(f"Survival rate: {departure_results['survived'].mean():.1%}")
        print(f"Mean KR: {departure_results['kr'].mean():.3f}")
        print(f"KR range: [{departure_results['kr'].min():.3f}, {departure_results['kr'].max():.3f}]")
        
        # Display detailed results table
        print("\n" + "="*60)
        print("DETAILED RESULTS")
        print("="*60)
        
        display_cols = ['repo_id', 'founder', 'kr', 'survived', 'survival_time']
        display_df = departure_results[display_cols].copy()
        display_df['survived'] = display_df['survived'].map({1: 'Yes', 0: 'No'})
        print(display_df.to_string(index=False))
        
        # Run statistical analyses
        print("\n" + "="*60)
        print("STATISTICAL ANALYSIS")
        print("="*60)
        
        # Cox proportional hazards model
        cox_results = run_survival_analysis(departure_results)
        
        # Bootstrap confidence intervals
        bootstrap_results = bootstrap_confidence_intervals(departure_results, N_BOOTSTRAP)
        
        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Plot 1: KR distribution
        axes[0].hist(departure_results['kr'], bins=10, edgecolor='black', alpha=0.7)
        axes[0].set_xlabel('Knowledge Redundancy (KR)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Distribution of Knowledge Redundancy')
        axes[0].axvline(departure_results['kr'].mean(), color='red', linestyle='--', 
                       label=f'Mean: {departure_results["kr"].mean():.3f}')
        axes[0].legend()
        
        # Plot 2: KR vs Survival (scatter)
        colors = ['green' if s == 1 else 'red' for s in departure_results['survived']]
        axes[1].scatter(departure_results['kr'], departure_results['survival_time'], 
                       c=colors, alpha=0.7, s=100)
        axes[1].set_xlabel('Knowledge Redundancy (KR)')
        axes[1].set_ylabel('Survival Time (days)')
        axes[1].set_title('KR vs Survival Time')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print hypothesis test results
        print("\n" + "="*60)
        print("HYPOTHESIS TEST RESULTS")
        print("="*60)
        
        if cox_results and cox_results.get('kr_squared_coef') is not None:
            kr2_coef = cox_results['kr_squared_coef']
            kr2_p = cox_results.get('kr_squared_p_value', 1.0)
            print(f"\nCox Proportional Hazards Model:")
            print(f"  KR^2 coefficient: {kr2_coef:.4f}")
            print(f"  p-value: {kr2_p:.4f}")
            if kr2_coef < 0 and kr2_p < 0.05:
                print(f"  RESULT: Inverted-U hypothesis SUPPORTED")
            else:
                print(f"  RESULT: Inverted-U hypothesis NOT supported")
        else:
            print("\nCox model could not be fitted (insufficient data or error)")
        
        if bootstrap_results:
            print(f"\nBootstrap Confidence Intervals:")
            print(f"  Samples: {bootstrap_results['bootstrap_n']}")
            print(f"  Mean survival diff: {bootstrap_results['survival_diff_mean']:.3f}")
            print(f"  95% CI: [{bootstrap_results['survival_diff_95ci'][0]:.3f}, {bootstrap_results['survival_diff_95ci'][1]:.3f}]")
    else:
        print("No repositories with founder departure found.")
else:
    print("No results to display.")
    results_df = pd.DataFrame()

## Conclusion

This notebook demonstrated the OSS founder departure survival analysis experiment. 

### Key Findings:
- The experiment processes commit records to detect founder departures and measure project survival
- Knowledge Redundancy (KR) is computed using a fallback approach (cosine similarity of file_count histograms)
- Statistical analyses include Cox proportional hazards models and Kaplan-Meier curves

### Limitations:
- Small sample size in this demo (scale up `N_BOOTSTRAP` and use full dataset for production)
- Fallback KR measure due to lack of file path data for Jaccard similarity
- All repos in the demo survived (no variation in survival outcome)

### Next Steps:
1. Scale up the analysis with more repositories
2. Increase bootstrap samples for more robust confidence intervals
3. Obtain file path data to compute true Jaccard similarity for KR